# Databricks Environment Validation

This notebook validates the Databricks environment used for the FinTech Data Platform.

In [0]:
print("Spark Session:", spark)
print("Spark Version:", spark.version)

In [0]:
data = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie")
]

df = spark.createDataFrame(data, ["id", "name"])

df.display()

In [0]:
df_filtered = df.filter(df.id > 1)

df_filtered.display()

## Observations

- Spark session is available.
- Spark version: 4.1.0
- Basic DataFrame creation and transformation work.
- `display()` successfully executes Spark transformations.
- `explain("formatted")` can be used to inspect the physical plan.
- PySpark RDD APIs are not supported on Serverless compute.

In [0]:
%sql

DESCRIBE DETAIL dbx_fintech_data_platform.bronze.transactions_autoloader_final

In [0]:
%sql

DESCRIBE HISTORY dbx_fintech_data_platform.bronze.transactions_autoloader_final

In [0]:
table_detail = spark.sql("""
    DESCRIBE DETAIL dbx_fintech_data_platform.bronze.transactions_autoloader_final
""")

display(
    table_detail.select("location", "format", "numFiles", "sizeInBytes")
)

In [0]:
%sql

CREATE OR REPLACE TABLE dbx_fintech_data_platform.silver.delta_learning_test (
    account_id STRING,
    account_type STRING,
    status STRING
)
USING DELTA;

In [0]:
%sql

INSERT INTO dbx_fintech_data_platform.silver.delta_learning_test
VALUES
    ('A001', 'SAVINGS', 'ACTIVE'),
    ('A002', 'CURRENT', 'ACTIVE'),
    ('A003', 'SAVINGS', 'ACTIVE');

In [0]:
%sql

UPDATE dbx_fintech_data_platform.silver.delta_learning_test
SET status = 'CLOSED'
WHERE account_id = 'A002';

In [0]:
%sql

DESCRIBE HISTORY dbx_fintech_data_platform.silver.delta_learning_test;

In [0]:
%sql

SELECT *
FROM dbx_fintech_data_platform.silver.delta_learning_test
VERSION AS OF 1;

In [0]:
from pyspark.sql import functions as F

data = [
    ("A001", "M001", 100),
    ("A002", "M001", 200),
    ("A003", "M002", 150),
    ("A004", "M001", 300),
    ("A005", "M002", 250),
    ("A006", "M003", 500),
]

df = spark.createDataFrame(
    data,
    ["account_id", "merchant_id", "amount"]
)

In [0]:
result = (
    df
    .groupBy("merchant_id")
    .agg(
        F.sum("amount").alias("total_amount")
    )
)

In [0]:
result.explain("formatted")

In [0]:
df = (
    spark.range(0, 1_000_000)
    .withColumn(
        "merchant_id",
        (F.col("id") % 100).cast("string")
    )
    .withColumn(
        "amount",
        (F.col("id") % 1000).cast("long")
    )
)

In [0]:
result = (
    df
    .groupBy("merchant_id")
    .agg(
        F.sum("amount").alias("total_amount")
    )
)

In [0]:
result.explain("formatted")

In [0]:
result.show()